## Topic modeling con Transfer learning (feature extraction) - domande di rinuncia agli studi

Le domande di rinuncia presentate dagli studenti contengono un campo "note" in cui è possibile inserire testo libero con i motivi dell'abbandono.

Il task è raggruppare in topic generali questi testi. Sfruttiamo NLP e i Transformers

In [1]:
#!pip install bertopic[all] sentence-transformers
#pip install tf-keras
##!pip install transformers torch sentencepiece

In [2]:
import pandas as pd
df = pd.read_csv(r'C:\Users\Utente\PycharmProjects\Data_mining_UPO\data\p04domchi2.csv', header=None, names=["testo"], sep=';', encoding="latin-1")
texts = df["testo"].fillna("").astype(str).tolist()
print("Check una riga se funziona:", texts[0])
print("Check numero record:", len(df))

Check una riga se funziona: Impossibilità a proseguire gli studi
Check numero record: 3319


In [3]:
# Rimuovo duplicati e righe troppo corte + lowering
# No preprocessing complesso perché SBERT / BERTopic già gestiscono (pare) stopword e tokenizzazione; così sembra anche più facilmente replicabile
df = df.drop_duplicates(subset="testo")
df = df[df["testo"].str.split().str.len() > 2]
df["testo"] = df["testo"].str.lower()
texts = df["testo"].tolist()

texts

['impossibilità a proseguire gli studi',
 'cambio percorso accedemico',
 'indisponibilità a pagare la ii rata del corso',
 'esami propedeutici lingue',
 'mancata disponibilità economica.',
 'spett.le università, \ncon mio rammarico, mi vedo costretto a rinunciare al percorso di studi scelto a causa degli improrogabili impegni di lavoro che non mi permettono di studiare in maniera adeguata.\nringrazandovi, porgo i più\ncordiali saluti, \nc. ',
 'incompatibilità con impegni lavorativi.',
 "non riesco a conciliare l'impegno dello studio con la mia attività lavorativa",
 'motivi di lavoro',
 'sono malata da tanto. sono rimasta iscritta nella speranza che le cose migliorassero, ma non è andata così. le mie condizioni sono molto peggiorate e le prospettive ormai sono solo di peggioramento.\npertanto è arrivato il momento per me di abbandonare gli studi.\nvi ringrazio, cordiali saluti.',
 "a chi di competenza,\n\nbuongiorno,\n\npochi giorni dopo l'immatricolazione, ho ottenuto un contratto di

In [4]:
from sentence_transformers import SentenceTransformer
#https://huggingface.co/sentence-transformers/distiluse-base-multilingual-cased-v2

# Modello BERT multilingue - uno dei più usati
model_name = "sentence-transformers/distiluse-base-multilingual-cased-v2"
sbert = SentenceTransformer(model_name)

# Calcolo embedding con sbert https://www.sbert.net/
embeddings = sbert.encode(texts, show_progress_bar=True)
print("Shape embeddings:", embeddings.shape)


C:\Users\Utente\PycharmProjects\Data_mining_UPO\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Batches: 100%|██████████| 70/70 [00:30<00:00,  2.26it/s]

Shape embeddings: (2237, 512)


In [5]:
print(embeddings.shape)   # (numero_testi, 512)
print(embeddings[0][:10]) # primi 10 valori del primo embedding

(2237, 512)
[-3.3089500e-02  3.5384533e-03  1.0899307e-02  1.6114295e-02
  7.6892860e-02 -4.2174044e-03  2.4098786e-02 -2.2410324e-02
 -2.2858584e-03  4.2111147e-05]


In [6]:
from sklearn.metrics.pairwise import cosine_similarity
# esempio che da provare che c'è su github
sim = cosine_similarity([embeddings[0]], [embeddings[1]])
print("Similarità tra frase 0 e frase 1:", sim[0][0])


Similarità tra frase 0 e frase 1: 0.27793556


In [1]:
import requests

url = "https://raw.githubusercontent.com/stopwords-iso/stopwords-it/master/stopwords-it.txt"
response = requests.get(url)

stopwords_italiane = set(response.text.splitlines())
print(len(stopwords_italiane))


632


In [8]:
from sklearn.feature_extraction.text import CountVectorizer
# faccio un listino aggiuntivo di stopwords che i modelli non colgono (i dont know why)
#custom_stopwords = ["non", "ho", "di", "per", "la", "il", "in", "al", "un", "una"]
vectorizer_model = CountVectorizer(stop_words=stopwords_italiane)

from bertopic import BERTopic
# https://maartengr.github.io/BERTopic/index.html#common
topic_model = BERTopic(language="italian",
                       embedding_model=sbert,
                       vectorizer_model=vectorizer_model,
                       # momento divertimento con i params :)
                       # metto nr_topics iperparam perchè se lancio così il modello genera 50+ topics
                       #nr_topics=5,
                       min_topic_size=50,
                       # se un giorno volessi semplificare tutto basta mettere il sentence trasformer qui
                       n_gram_range=(1, 2) #usa tutte le sequenze da 1, 2 parole come possibili keyword dei topic
)
topics, probs = topic_model.fit_transform(texts, embeddings)

# Assegna topics al dataframe
df["topics"] = topics
#df.to_csv("rinunce_topics3.csv", index=False)

# Report rapido
topic_info = topic_model.get_topic_info()
print(topic_info.head())


   Topic  Count                                        Name  \
0     -1    256               -1_inizio_salute_studi_milano   
1      0    752               0_studi_percorso_corso_deciso   
2      1    428  1_altra_iscrizione_ateneo_immatricolazione   
3      2    323     2_motivi_impossibilità_lavoro_personali   
4      3    236           3_cambio_facoltà_corso_università   

                                      Representation  \
0  [inizio, salute, studi, milano, problemi, moti...   
1  [studi, percorso, corso, deciso, to, universit...   
2  [altra, iscrizione, ateneo, immatricolazione, ...   
3  [motivi, impossibilità, lavoro, personali, lav...   
4  [cambio, facoltà, corso, università, studi, ca...   

                                 Representative_Docs  
0  [problemi di salute, rinuncia per motivi di sa...  
1  [ho deciso di intraprendere un altro percorso ...  
2  [iscrizione presso un'altra università, iscriz...  
3  [impossibilità nel proseguire con gli studi pe...  
4  [cambi

In [9]:
# il -1 è il cluster degli outliers
topic_info

,Topic,Count,Name,Representation,Representative_Docs
0,-1,256,-1_inizio_salute_studi_milano,"[inizio, salute, studi, milano, problemi, moti...","[problemi di salute, rinuncia per motivi di sa..."
1,0,752,0_studi_percorso_corso_deciso,"[studi, percorso, corso, deciso, to, universit...",[ho deciso di intraprendere un altro percorso ...
2,1,428,1_altra_iscrizione_ateneo_immatricolazione,"[altra, iscrizione, ateneo, immatricolazione, ...","[iscrizione presso un'altra università, iscriz..."
3,2,323,2_motivi_impossibilità_lavoro_personali,"[motivi, impossibilità, lavoro, personali, lav...",[impossibilità nel proseguire con gli studi pe...
4,3,236,3_cambio_facoltà_corso_università,"[cambio, facoltà, corso, università, studi, ca...","[cambio di corso, cambio del corso , cambio di..."
5,4,94,4_scelta_sbagliata_percorso_sbagliato,"[scelta, sbagliata, percorso, sbagliato, studi...","[scelta di un altro percorso, scelta altro per..."
6,5,85,5_medicina_chirurgia_facoltà_presso,"[medicina, chirurgia, facoltà, presso, immatri...",[immatricolazione a medicina e chirurgia press...
7,6,63,6_trasferimento_ateneo_passaggio_presso,"[trasferimento, ateneo, passaggio, presso, alt...","[trasferimento ad un altro ateneo, trasferimen..."


In [10]:
topic_model.get_topic_freq()

,Topic,Count
3,0,752
4,1,428
0,2,323
2,-1,256
1,3,236
5,4,94
6,5,85
7,6,63


In [11]:
topic_model.generate_topic_labels()

['-1_inizio_salute_studi',
 '0_studi_percorso_corso',
 '1_altra_iscrizione_ateneo',
 '2_motivi_impossibilità_lavoro',
 '3_cambio_facoltà_corso',
 '4_scelta_sbagliata_percorso',
 '5_medicina_chirurgia_facoltà',
 '6_trasferimento_ateneo_passaggio']

In [12]:
topic_model.get_params()

# questo modello usa di default HDBSCAN per clusterizzare i topic dopo l'embedding!

{'calculate_probabilities': False,
 'ctfidf_model': ClassTfidfTransformer(),
 'embedding_model': <bertopic.backend._sentencetransformers.SentenceTransformerBackend at 0x183b062af10>,
 'hdbscan_model': HDBSCAN(min_cluster_size=50, prediction_data=True),
 'language': None,
 'low_memory': False,
 'min_topic_size': 50,
 'n_gram_range': (1, 2),
 'nr_topics': None,
 'representation_model': None,
 'seed_topic_list': None,
 'top_n_words': 10,
 'umap_model': UMAP(angular_rp_forest=True, low_memory=False, metric='cosine', min_dist=0.0, n_components=5, tqdm_kwds={'bar_format': '{desc}: {percentage:3.0f}%| {bar} {n_fmt}/{total_fmt} [{elapsed}]', 'desc': 'Epochs completed', 'disable': True}),
 'vectorizer_model': CountVectorizer(stop_words=['a', 'abbastanza', 'abbia', 'abbiamo', 'abbiano',
                             'abbiate', 'accidenti', 'ad', 'adesso', 'affinché',
                             'agl', 'agli', 'ahime', 'ahimè', 'ai', 'al',
                             'alcuna', 'alcuni', 'alcuno'

In [13]:
topic_model.visualize_topics()

In [14]:
topic_model.get_representative_docs()

{-1: ['problemi di salute',
  'rinuncia per motivi di salute',
  'immatricolazione presso università degli studi di milano.'],
 0: ['ho deciso di intraprendere un altro percorso di studi',
  'ho deciso di non proseguire il corso dei miei studi',
  "ho deciso di intraprendere un altro percorso di studi in un'altra università"],
 1: ["iscrizione presso un'altra università",
  'iscrizione presso altra università',
  "iscrizione presso un altro corso di laurea di un'altra università"],
 2: ['impossibilità nel proseguire con gli studi per motivi personali.',
  'impossibilità a proseguire gli studi per motivi personali',
  'motivi personali. ho trovato lavoro'],
 3: ['cambio di corso', 'cambio del corso ', 'cambio di corso'],
 4: ['scelta di un altro percorso',
  'scelta altro percorso di studi ',
  'scelta altro corso'],
 5: ['immatricolazione a medicina e chirurgia presso altro ateneo',
  'immatricolazione alla facoltà di medicina e chirurgia presso un altro ateneo.',
  'iscrizione alla fa

In [15]:
#df.to_csv("rinunce_topics3.csv", index=False)